In [ ]:
import pandas as pd
import numpy as np
import requests
import warnings
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')

print("Librerías cargadas correctamente")

Librerías cargadas correctamente


In [ ]:
# ====================== CARGA DE DATOS LOCALES ======================
LOCAL_FILE = "2018DAGMASISNO2.csv"   # ← Cambia el nombre si es diferente

df_local = pd.read_csv(LOCAL_FILE)

print("Archivo local cargado!")
print("Dimensiones:", df_local.shape)
print("Columnas:", df_local.columns.tolist())

df_local.head()

Archivo local cargado!
Dimensiones: (6494, 4)
Columnas: ['Estacion', 'Fecha inicial', 'Fecha final', 'NO2']


,Estacion,Fecha inicial,Fecha final,NO2
0,UNIVERSIDAD DEL VALLE,2018-12-31 20:00,2018-12-31 20:59,33.3
1,UNIVERSIDAD DEL VALLE,2018-12-31 10:00,2018-12-31 10:59,7.2
2,UNIVERSIDAD DEL VALLE,2018-12-31 09:00,2018-12-31 09:59,5.7
3,UNIVERSIDAD DEL VALLE,2018-12-31 08:00,2018-12-31 08:59,6.2
4,UNIVERSIDAD DEL VALLE,2018-12-31 06:00,2018-12-31 06:59,4.0


In [ ]:
# Limpieza
df_local.columns = ['Estacion', 'Fecha_inicial', 'Fecha_final', 'NO2']  # Ajusta si las columnas son diferentes

# Convertir fechas
df_local['Fecha_inicial'] = pd.to_datetime(df_local['Fecha_inicial'], errors='coerce')
df_local['Fecha_final']   = pd.to_datetime(df_local['Fecha_final'], errors='coerce')

# Usaremos Fecha_inicial como timestamp principal
df_local['datetime'] = df_local['Fecha_inicial']
df_local['date'] = df_local['datetime'].dt.date
df_local['hour'] = df_local['datetime'].dt.hour
df_local['year'] = df_local['datetime'].dt.year
df_local['month'] = df_local['datetime'].dt.month
df_local['dayofweek'] = df_local['datetime'].dt.dayofweek
df_local['weekday_name'] = df_local['datetime'].dt.day_name()

# Convertir NO2 a numérico
df_local['NO2'] = pd.to_numeric(df_local['NO2'], errors='coerce')

# Agregar columnas PM (como hizo tu compañero)
df_local['PM1.0'] = np.nan
df_local['PM2.5'] = np.nan

print(" Datos 2018 limpios")
print(df_local['Estacion'].value_counts())
df_local.head()

 Datos 2018 limpios
Estacion
UNIVERSIDAD DEL VALLE    6494
Name: count, dtype: int64


,Estacion,Fecha_inicial,Fecha_final,NO2,datetime,date,hour,year,month,dayofweek,weekday_name,PM1.0,PM2.5
0,UNIVERSIDAD DEL VALLE,2018-12-31 20:00:00,2018-12-31 20:59:00,33.3,2018-12-31 20:00:00,2018-12-31,20,2018,12,0,Monday,NaN,NaN
1,UNIVERSIDAD DEL VALLE,2018-12-31 10:00:00,2018-12-31 10:59:00,7.2,2018-12-31 10:00:00,2018-12-31,10,2018,12,0,Monday,NaN,NaN
2,UNIVERSIDAD DEL VALLE,2018-12-31 09:00:00,2018-12-31 09:59:00,5.7,2018-12-31 09:00:00,2018-12-31,9,2018,12,0,Monday,NaN,NaN
3,UNIVERSIDAD DEL VALLE,2018-12-31 08:00:00,2018-12-31 08:59:00,6.2,2018-12-31 08:00:00,2018-12-31,8,2018,12,0,Monday,NaN,NaN
4,UNIVERSIDAD DEL VALLE,2018-12-31 06:00:00,2018-12-31 06:59:00,4.0,2018-12-31 06:00:00,2018-12-31,6,2018,12,0,Monday,NaN,NaN


In [ ]:
APP_TOKEN = "vsKf3GrMJtVh3cuyjoL9xiuBFLFhIjkWSYZ5"   # Tu token secreto

def descargar_datos_gov(limit=50000, offset=0):
    url = "https://www.datos.gov.co/resource/g4t8-zkc3.json"

    headers = {"0uwNTnqz1uvTu3j4ytEADYONR": APP_TOKEN}

    params = {
        "$limit": limit,
        "$offset": offset,
        "$order": "med_fecha_inicio ASC",
        "$where": "med_fecha_inicio >= '2018-01-01' AND med_fecha_inicio <= '2022-12-31'"
    }

    print(f"Descargando registros {offset} a {offset+limit}...")
    response = requests.get(url, headers=headers, params=params)

    if response.status_code == 200:
        data = response.json()
        df = pd.DataFrame(data)
        print(f"Descargados {len(df)} registros")
        return df
    else:
        print("Error:", response.status_code)
        print(response.text[:500])
        return None

# Descargar (puede tomar tiempo, lo hacemos en bloques si es necesario)
df_gov = descargar_datos_gov(limit=100000)

Descargando registros 0 a 100000...
Descargados 100000 registros


In [ ]:
# Explorar estructura de los datos del gobierno
print("Columnas disponibles en datos.gov.co:")
print(df_gov.columns.tolist())

print("\nShape:", df_gov.shape)
print("\nPrimeras 5 filas:")
display(df_gov.head())

print("\nEstaciones / Municipios disponibles:")
if 'municipio' in df_gov.columns:
    print(df_gov['municipio'].value_counts().head(10))
elif 'estacion' in df_gov.columns:
    print(df_gov['estacion'].value_counts().head(10))

print("\nTipos de contaminantes:")
if 'nombre_contaminante' in df_gov.columns:
    print(df_gov['nombre_contaminante'].value_counts())
elif 'diametro_aerodinamico' in df_gov.columns:
    print(df_gov['diametro_aerodinamico'].value_counts())

Columnas disponibles en datos.gov.co:
['estacion_id', 'nombre_fgda', 'nombre_est', 'msfl_code', 'med_concentracion_estandar', 'med_fecha_inicio', 'med_fecha_final', 'latitud', 'longitud', 'altitud', 'nombre_unidad', 'sigla_unidad', 'duraci_n', 'codigo_departamento', 'departamento', 'codigo_municipio', 'municipio', 'tipo_estacion']

Shape: (100000, 18)

Primeras 5 filas:


,estacion_id,nombre_fgda,nombre_est,msfl_code,med_concentracion_estandar,med_fecha_inicio,med_fecha_final,latitud,longitud,altitud,nombre_unidad,sigla_unidad,duraci_n,codigo_departamento,departamento,codigo_municipio,municipio,tipo_estacion
0,8204,CDMB,CIUDADELA,HAire10,92,2020-01-01T00:00:00.000,2020-01-01T01:00:00.000,7.105925,-73.123691666667,835,Porcentaje,%,60,68,Santander,68001,Bucaramanga,Fija
1,8204,CDMB,CIUDADELA,PLiquida,0,2020-01-01T00:00:00.000,2020-01-01T01:00:00.000,7.105925,-73.123691666667,835,Milimetros,mm,60,68,Santander,68001,Bucaramanga,Fija
2,31661,CORANTIOQUIA,IGLE,PLiquida,0,2020-01-01T00:00:00.000,2020-01-01T01:00:00.000,6.399528,-75.438028,1408,Milimetros,mm,60,5,Antioquia,5308,Girardota,Indicativa
3,31812,AMB,LAGOS DEL CACIQUE,RGlobal,0,2020-01-01T00:00:00.000,2020-01-01T01:00:00.000,7.100051,-73.104175,887,Vatios hora por metro cuadrado,Wh/m2,60,68,Santander,68001,Bucaramanga,Fija
4,30004,DAGMA,ERA OBRERO,PLiquida,0.25,2020-01-01T00:00:00.000,2020-01-01T01:00:00.000,3.457317,-76.506539,968,Milimetros,mm,60,76,Valle del Cauca,76001,Santiago de Cali,Fija



Estaciones / Municipios disponibles:
municipio
Bogotá, D.C.        23284
Medellín            14076
Santiago de Cali     9818
Bucaramanga          6608
Barranquilla         6137
Girardota            4593
Sabaneta             4284
Itagüí               3693
Bello                2716
Sogamoso             2575
Name: count, dtype: int64

Tipos de contaminantes:


In [ ]:
# Filtrar datos relevantes
df_gov_clean = df_gov.copy()

# Convertir fecha
if 'med_fecha_inicio' in df_gov_clean.columns:
    df_gov_clean['datetime'] = pd.to_datetime(df_gov_clean['med_fecha_inicio'], errors='coerce')
elif 'fecha' in df_gov_clean.columns:
    df_gov_clean['datetime'] = pd.to_datetime(df_gov_clean['fecha'], errors='coerce')

# Crear columnas temporales
df_gov_clean['date'] = df_gov_clean['datetime'].dt.date
df_gov_clean['hour'] = df_gov_clean['datetime'].dt.hour
df_gov_clean['year'] = df_gov_clean['datetime'].dt.year
df_gov_clean['month'] = df_gov_clean['datetime'].dt.month
df_gov_clean['dayofweek'] = df_gov_clean['datetime'].dt.dayofweek

# Filtrar por zona de interés (ajustaremos según lo que muestre)
zonas_interes = ['CALI', 'SANTIAGO DE CALI', 'YUMBO', 'VALLE DEL CAUCA']
if 'municipio' in df_gov_clean.columns:
    df_gov_filtered = df_gov_clean[df_gov_clean['municipio'].isin(zonas_interes)]
else:
    df_gov_filtered = df_gov_clean

print("Registros después de filtrar zona:", len(df_gov_filtered))
print(df_gov_filtered['municipio'].value_counts() if 'municipio' in df_gov_filtered.columns else "Sin columna municipio")

Registros después de filtrar zona: 0
Series([], Name: count, dtype: int64)


In [ ]:
# Buscar estaciones relevantes en Cali, Yumbo o Valle
print("Buscando estaciones en Cali / Yumbo...")

# Todas las estaciones únicas
estaciones = df_gov['nombre_est'].value_counts().head(20)
print("\nEstaciones más frecuentes:")
print(estaciones)

print("\nMunicipios disponibles:")
print(df_gov['municipio'].value_counts().head(15))

# Buscar específicamente Cali o Yumbo
cali_yumbo = df_gov[df_gov['municipio'].str.contains('CALI|YUMBO|VALLE', case=False, na=False)]
print(f"\nRegistros encontrados en Cali/Yumbo/Valle: {len(cali_yumbo)}")

if len(cali_yumbo) > 0:
    print("\nEstaciones en zona de interés:")
    print(cali_yumbo['nombre_est'].value_counts())
    print("\nMunicipios:")
    print(cali_yumbo['municipio'].value_counts())

Buscando estaciones en Cali / Yumbo...

Estaciones más frecuentes:
nombre_est
TUNAL                         2428
C. ALTO RENDIMIENTO           2384
KENNEDY                       2350
POLICÍA                       2257
LAS FERIAS                    2200
TRES AVE MARÍAS               2178
GUAYMARAL                     2127
U. NAL - NÚCLEO EL VOLADOR    2102
S.O.S ABURRÁ NORTE            2102
EL RECREO                     2040
PUENTE ARANDA                 1980
NAZARETH-JAC                  1968
CIUDADELA AMB-CDMB            1898
POLITÉCNICO JIC               1897
TRÁFICO SUR                   1887
FONTIBÓN                      1854
COMPARTIR                     1832
I.E. CONCEJO MUNICIPAL        1778
CASA JUSTICIA                 1715
MÓVIL                         1702
Name: count, dtype: int64

Municipios disponibles:
municipio
Bogotá, D.C.        23284
Medellín            14076
Santiago de Cali     9818
Bucaramanga          6608
Barranquilla         6137
Girardota            4593
Saban

In [ ]:
# ==================== FILTRADO CORREGIDO ====================
df_gov_clean = df_gov.copy()

# Convertir fecha
df_gov_clean['datetime'] = pd.to_datetime(df_gov_clean['med_fecha_inicio'], errors='coerce')

# Columnas temporales
df_gov_clean['date'] = df_gov_clean['datetime'].dt.date
df_gov_clean['hour'] = df_gov_clean['datetime'].dt.hour
df_gov_clean['year'] = df_gov_clean['datetime'].dt.year
df_gov_clean['month'] = df_gov_clean['datetime'].dt.month

# Filtrar zona de interés (más amplio)
mask = (
    df_gov_clean['municipio'].str.contains('CALI|YUMBO|VALLE', case=False, na=False) |
    df_gov_clean['nombre_est'].str.contains('CALI|YUMBO|UNIVERSIDAD|UNIVALLE', case=False, na=False)
)

df_gov_filtered = df_gov_clean[mask].copy()

print(f"Registros después de filtrar zona: {len(df_gov_filtered)}")

if len(df_gov_filtered) > 0:
    print("\nEstaciones encontradas:")
    print(df_gov_filtered['nombre_est'].value_counts())
    print("\nMunicipios:")
    print(df_gov_filtered['municipio'].value_counts())

    # Ver contaminantes en esta zona
    print("\nUnidades de medida:")
    print(df_gov_filtered['nombre_unidad'].value_counts())

Registros después de filtrar zona: 10170

Estaciones encontradas:
nombre_est
COMPARTIR                1832
UNIVERSIDAD DEL VALLE    1605
CAÑAVERALEJO             1563
PANCE                    1343
LA FLORA                 1227
ERA OBRERO               1094
LA ERMITA                 748
BASE AÉREA                406
ACOPI                     340
BOMBEROS                    6
SEMINARIO                   6
Name: count, dtype: int64

Municipios:
municipio
Santiago de Cali    9818
Yumbo                340
Valledupar            12
Name: count, dtype: int64

Unidades de medida:
nombre_unidad
Microgramos por metro cúbico      2773
Milimetros                        1218
Metros por segundo                1217
Grados sexadecimales              1216
Grados Celsius                    1215
Porcentaje                        1015
Vatios hora por metro cuadrado     904
Milimetros de mercurio             612
Name: count, dtype: int64


In [ ]:
# ==================== EXPLORAR CONTAMINANTES EN ZONA DE INTERÉS ====================
df_cali = df_gov_filtered.copy()

print("Contaminantes / Mediciones en Cali/Yumbo:")
print(df_cali['nombre_unidad'].value_counts())

print("\n Tipos de mediciones (sigla o nombre):")
if 'msfl_code' in df_cali.columns:
    print(df_cali['msfl_code'].value_counts().head(20))

print("\nEstaciones con más datos de concentración (Microgramos/m³):")
micro = df_cali[df_cali['nombre_unidad'].str.contains('Microgramos', na=False)]
print(micro['nombre_est'].value_counts().head(10))

print("\nMuestras de datos de Universidad del Valle:")
display(micro[micro['nombre_est'] == 'UNIVERSIDAD DEL VALLE'].head(8))

Contaminantes / Mediciones en Cali/Yumbo:
nombre_unidad
Microgramos por metro cúbico      2773
Milimetros                        1218
Metros por segundo                1217
Grados sexadecimales              1216
Grados Celsius                    1215
Porcentaje                        1015
Vatios hora por metro cuadrado     904
Milimetros de mercurio             612
Name: count, dtype: int64

 Tipos de mediciones (sigla o nombre):
msfl_code
PM10        1231
PLiquida    1218
VViento     1217
DViento     1216
TAire2      1215
HAire2      1015
RGlobal      904
PM2.5        752
P            612
O3           418
SO2          372
Name: count, dtype: int64

Estaciones con más datos de concentración (Microgramos/m³):
nombre_est
COMPARTIR                506
LA FLORA                 412
LA ERMITA                343
ACOPI                    340
ERA OBRERO               282
PANCE                    251
CAÑAVERALEJO             239
BASE AÉREA               204
UNIVERSIDAD DEL VALLE    184
BOMBEROS  

,estacion_id,nombre_fgda,nombre_est,msfl_code,med_concentracion_estandar,med_fecha_inicio,med_fecha_final,latitud,longitud,altitud,...,codigo_departamento,departamento,codigo_municipio,municipio,tipo_estacion,datetime,date,hour,year,month
5890,8291,DAGMA,UNIVERSIDAD DEL VALLE,PM2.5,17,2020-01-01T12:00:00.000,2020-01-01T13:00:00.000,3.377911,-76.533811,985,...,76,Valle del Cauca,76001,Santiago de Cali,Fija,2020-01-01 12:00:00,2020-01-01,12,2020,1
6302,8291,DAGMA,UNIVERSIDAD DEL VALLE,PM2.5,14,2020-01-01T13:00:00.000,2020-01-01T14:00:00.000,3.377911,-76.533811,985,...,76,Valle del Cauca,76001,Santiago de Cali,Fija,2020-01-01 13:00:00,2020-01-01,13,2020,1
6942,8291,DAGMA,UNIVERSIDAD DEL VALLE,PM2.5,20,2020-01-01T14:00:00.000,2020-01-01T15:00:00.000,3.377911,-76.533811,985,...,76,Valle del Cauca,76001,Santiago de Cali,Fija,2020-01-01 14:00:00,2020-01-01,14,2020,1
7518,8291,DAGMA,UNIVERSIDAD DEL VALLE,PM2.5,32,2020-01-01T15:00:00.000,2020-01-01T16:00:00.000,3.377911,-76.533811,985,...,76,Valle del Cauca,76001,Santiago de Cali,Fija,2020-01-01 15:00:00,2020-01-01,15,2020,1
7895,8291,DAGMA,UNIVERSIDAD DEL VALLE,PM2.5,25,2020-01-01T16:00:00.000,2020-01-01T17:00:00.000,3.377911,-76.533811,985,...,76,Valle del Cauca,76001,Santiago de Cali,Fija,2020-01-01 16:00:00,2020-01-01,16,2020,1
8379,8291,DAGMA,UNIVERSIDAD DEL VALLE,PM2.5,24,2020-01-01T17:00:00.000,2020-01-01T18:00:00.000,3.377911,-76.533811,985,...,76,Valle del Cauca,76001,Santiago de Cali,Fija,2020-01-01 17:00:00,2020-01-01,17,2020,1
9034,8291,DAGMA,UNIVERSIDAD DEL VALLE,PM2.5,31,2020-01-01T18:00:00.000,2020-01-01T19:00:00.000,3.377911,-76.533811,985,...,76,Valle del Cauca,76001,Santiago de Cali,Fija,2020-01-01 18:00:00,2020-01-01,18,2020,1
9427,8291,DAGMA,UNIVERSIDAD DEL VALLE,PM2.5,26,2020-01-01T19:00:00.000,2020-01-01T20:00:00.000,3.377911,-76.533811,985,...,76,Valle del Cauca,76001,Santiago de Cali,Fija,2020-01-01 19:00:00,2020-01-01,19,2020,1


In [ ]:
# ==================== PREPARACIÓN DE DATOS COMBINADOS ====================

# 1. Datos locales (2018)
df_local_cali = df_local[df_local['Estacion'].str.contains('UNIVERSIDAD DEL VALLE', na=False)].copy()
df_local_cali = df_local_cali.rename(columns={'NO2': 'NO2_local'})

# 2. Datos del gobierno para Universidad del Valle
df_gov_univalle = df_gov_filtered[df_gov_filtered['nombre_est'] == 'UNIVERSIDAD DEL VALLE'].copy()

# Convertir concentración
df_gov_univalle['concentracion'] = pd.to_numeric(df_gov_univalle['med_concentracion_estandar'], errors='coerce')

print(f"Registros locales UNIVALLE: {len(df_local_cali)}")
print(f"Registros gov UNIVALLE: {len(df_gov_univalle)}")

# Ver qué contaminantes tiene el gobierno en UNIVALLE
print("\nContaminantes en Universidad del Valle (gov):")
if 'msfl_code' in df_gov_univalle.columns:
    print(df_gov_univalle['msfl_code'].value_counts())

Registros locales UNIVALLE: 6494
Registros gov UNIVALLE: 1605

Contaminantes en Universidad del Valle (gov):
msfl_code
PLiquida    203
RGlobal     203
HAire2      203
VViento     203
P           203
TAire2      203
DViento     203
PM2.5       184
Name: count, dtype: int64


In [21]:
# ==================== CREAR DATASET COMBINADO ====================

# 1. Preparar datos locales (NO2 2018)
df_local_univalle = df_local[df_local['Estacion'].str.contains('UNIVERSIDAD DEL VALLE', na=False)].copy()
df_local_univalle['datetime'] = pd.to_datetime(df_local_univalle['Fecha_inicial'], errors='coerce')
df_local_univalle = df_local_univalle.rename(columns={'NO2': 'NO2'})

# 2. Preparar datos PM2.5 del gobierno
df_pm25 = df_gov_filtered[
    (df_gov_filtered['nombre_est'] == 'UNIVERSIDAD DEL VALLE') &
    (df_gov_filtered['msfl_code'] == 'PM2.5')
].copy()

df_pm25['datetime'] = pd.to_datetime(df_pm25['med_fecha_inicio'], errors='coerce')
df_pm25['PM2.5'] = pd.to_numeric(df_pm25['med_concentracion_estandar'], errors='coerce')

# Seleccionar solo columnas necesarias
df_pm25 = df_pm25[['datetime', 'PM2.5']].copy()

print(f"NO2 Local (2018): {len(df_local_univalle)} registros")
print(f"PM2.5 Gov: {len(df_pm25)} registros")

# 3. Unir ambos datasets por fecha/hora
df_combined = pd.merge(
    df_local_univalle[['datetime', 'NO2', 'hour', 'month', 'dayofweek']],
    df_pm25,
    on='datetime',
    how='outer'
)

df_combined = df_combined.sort_values('datetime').reset_index(drop=True)

# Agregar PM1.0 (por ahora vacío, como hizo tu compañero)
df_combined['PM1.0'] = np.nan

print(f"\nDataset combinado total: {len(df_combined)} registros")
print("Rango de fechas:", df_combined['datetime'].min(), "→", df_combined['datetime'].max())
df_combined.head(10)

NO2 Local (2018): 6494 registros
PM2.5 Gov: 184 registros

Dataset combinado total: 6678 registros
Rango de fechas: 2018-01-01 00:00:00 → 2020-01-09 10:00:00


,datetime,NO2,hour,month,dayofweek,PM2.5,PM1.0
0,2018-01-01 00:00:00,12.732025,0.0,1.0,0.0,NaN,NaN
1,2018-01-01 01:00:00,15.029816,1.0,1.0,0.0,NaN,NaN
2,2018-01-01 02:00:00,16.084540,2.0,1.0,0.0,NaN,NaN
3,2018-01-01 03:00:00,8.531963,3.0,1.0,0.0,NaN,NaN
4,2018-01-01 04:00:00,7.835092,4.0,1.0,0.0,NaN,NaN
5,2018-01-01 05:00:00,10.396564,5.0,1.0,0.0,NaN,NaN
6,2018-01-01 06:00:00,7.326564,6.0,1.0,0.0,NaN,NaN
7,2018-01-01 07:00:00,5.574969,7.0,1.0,0.0,NaN,NaN
8,2018-01-01 08:00:00,2.316626,8.0,1.0,0.0,NaN,NaN
9,2018-01-01 09:00:00,2.429632,9.0,1.0,0.0,NaN,NaN


In [ ]:
print("Distribución por año:")
print(df_combined['datetime'].dt.year.value_counts().sort_index())

print("\nDatos completos (NO2 + PM2.5):")
print(df_combined[['NO2', 'PM2.5']].count())

# Correlación entre NO2 y PM2.5 (donde ambos existen)
corr = df_combined[['NO2', 'PM2.5']].corr()
print("\nCorrelación NO2 vs PM2.5:")
print(corr)

Distribución por año:
datetime
2018    6494
2020     184
Name: count, dtype: int64

Datos completos (NO2 + PM2.5):
NO2      6494
PM2.5     184
dtype: int64

Correlación NO2 vs PM2.5:
       NO2  PM2.5
NO2    1.0    NaN
PM2.5  NaN    1.0


In [22]:
import pandas as pd
import numpy as np

# ==================== DATASET COMBINADO MEJORADO ====================

# Datos locales NO2 (2018)
df_no2 = df_local[df_local['Estacion'].str.contains('UNIVERSIDAD DEL VALLE', na=False)].copy()
df_no2['datetime'] = pd.to_datetime(df_no2['Fecha_inicial'], errors='coerce')
df_no2 = df_no2[['datetime', 'NO2']].copy()
df_no2.set_index('datetime', inplace=True)
df_no2 = df_no2.resample('H').mean()   # Aseguramos frecuencia horaria

# Datos PM2.5 del gobierno
df_pm25 = df_gov_filtered[
    (df_gov_filtered['nombre_est'] == 'UNIVERSIDAD DEL VALLE') &
    (df_gov_filtered['msfl_code'] == 'PM2.5')
].copy()

df_pm25['datetime'] = pd.to_datetime(df_pm25['med_fecha_inicio'], errors='coerce')
df_pm25['PM2.5'] = pd.to_numeric(df_pm25['med_concentracion_estandar'], errors='coerce')
df_pm25 = df_pm25[['datetime', 'PM2.5']].copy()
df_pm25.set_index('datetime', inplace=True)
df_pm25 = df_pm25.resample('H').mean()

# Unir todo en un solo dataframe con índice horario
df_combined = pd.concat([df_no2, df_pm25], axis=1)

# Agregar PM1.0 (placeholder)
df_combined['PM1.0'] = np.nan

# Features temporales
df_combined = df_combined.reset_index()
df_combined['hour'] = df_combined['datetime'].dt.hour
df_combined['month'] = df_combined['datetime'].dt.month
df_combined['dayofweek'] = df_combined['datetime'].dt.dayofweek
df_combined['year'] = df_combined['datetime'].dt.year
df_combined['is_weekend'] = df_combined['dayofweek'].isin([5,6]).astype(int)

print("Dataset final:")
print(df_combined.info())
print("\nRegistros por año:")
print(df_combined['year'].value_counts().sort_index())
print("\nDisponibilidad de variables:")
print(df_combined[['NO2', 'PM2.5', 'PM1.0']].count())

Dataset final:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8948 entries, 0 to 8947
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   datetime    8948 non-null   datetime64[ns]
 1   NO2         6494 non-null   float64       
 2   PM2.5       184 non-null    float64       
 3   PM1.0       0 non-null      float64       
 4   hour        8948 non-null   int32         
 5   month       8948 non-null   int32         
 6   dayofweek   8948 non-null   int32         
 7   year        8948 non-null   int32         
 8   is_weekend  8948 non-null   int64         
dtypes: datetime64[ns](1), float64(3), int32(4), int64(1)
memory usage: 489.5 KB
None

Registros por año:
year
2018    8757
2020     191
Name: count, dtype: int64

Disponibilidad de variables:
NO2      6494
PM2.5     184
PM1.0       0
dtype: int64


In [23]:
from sklearn.model_selection import train_test_split
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Preparar features
features = ['hour', 'month', 'dayofweek', 'is_weekend', 'year']

# Usar solo filas con NO2 (para entrenar)
train_df = df_combined[df_combined['NO2'].notna()].copy()

X = train_df[features]
y = train_df['NO2']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = xgb.XGBRegressor(
    n_estimators=800,
    learning_rate=0.05,
    max_depth=10,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

model.fit(X_train, y_train)

# Evaluación
preds = model.predict(X_test)
print("MAE:", mean_absolute_error(y_test, preds))
print("RMSE:", np.sqrt(mean_squared_error(y_test, preds)))

MAE: 5.639587438376807
RMSE: 7.854530624445389


In [25]:
# ==================== PREDICCIÓN 2019-2022 ====================

# Crear rango de fechas futuro (cada hora)
future_dates = pd.date_range(start='2019-01-01 00:00:00',
                             end='2022-12-31 23:00:00',
                             freq='H')

df_future = pd.DataFrame({'datetime': future_dates})

# Agregar features temporales
df_future['hour'] = df_future['datetime'].dt.hour
df_future['month'] = df_future['datetime'].dt.month
df_future['dayofweek'] = df_future['datetime'].dt.dayofweek
df_future['year'] = df_future['datetime'].dt.year
df_future['is_weekend'] = df_future['dayofweek'].isin([5,6]).astype(int)

# Predecir NO2
features = ['hour', 'month', 'dayofweek', 'is_weekend', 'year']
df_future['NO2_pred'] = model.predict(df_future[features])

# Agregar columnas PM (por ahora con valores promedio o NaN)
df_future['PM2.5'] = np.nan
df_future['PM1.0'] = np.nan

print("Predicciones generadas!")
print("Rango:", df_future['datetime'].min(), "→", df_future['datetime'].max())
print("Shape:", df_future.shape)
df_future.head()

Predicciones generadas!
Rango: 2019-01-01 00:00:00 → 2022-12-31 23:00:00
Shape: (35064, 9)


,datetime,hour,month,dayofweek,year,is_weekend,NO2_pred,PM2.5,PM1.0
0,2019-01-01 00:00:00,0,1,1,2019,0,11.514879,NaN,NaN
1,2019-01-01 01:00:00,1,1,1,2019,0,5.980488,NaN,NaN
2,2019-01-01 02:00:00,2,1,1,2019,0,4.259348,NaN,NaN
3,2019-01-01 03:00:00,3,1,1,2019,0,2.748185,NaN,NaN
4,2019-01-01 04:00:00,4,1,1,2019,0,3.991897,NaN,NaN


In [26]:
# Unir datos reales (2018 + 2020 PM2.5) con predicciones
df_historico = df_combined[['datetime', 'NO2', 'PM2.5', 'PM1.0']].copy()
df_historico['tipo'] = 'Real'

df_pred = df_future[['datetime', 'NO2_pred', 'PM2.5', 'PM1.0']].copy()
df_pred = df_pred.rename(columns={'NO2_pred': 'NO2'})
df_pred['tipo'] = 'Predicho'

df_final = pd.concat([df_historico, df_pred], ignore_index=True)
df_final = df_final.sort_values('datetime').reset_index(drop=True)

print("Dataset final creado!")
print("Total registros:", len(df_final))
print("Por año:")
print(df_final['datetime'].dt.year.value_counts().sort_index())

Dataset final creado!
Total registros: 44012
Por año:
datetime
2018    8757
2019    8760
2020    8975
2021    8760
2022    8760
Name: count, dtype: int64


In [27]:
# Guardar resultados
df_final.to_csv('NO2_Predicciones_2018_2022.csv', index=False)
df_future.to_csv('Solo_Predicciones_2019_2022.csv', index=False)

print("Archivos guardados en Google Colab:")
print("- NO2_Predicciones_2018_2022.csv  (todo)")
print("- Solo_Predicciones_2019_2022.csv (solo predicciones)")

Archivos guardados en Google Colab:
- NO2_Predicciones_2018_2022.csv  (todo)
- Solo_Predicciones_2019_2022.csv (solo predicciones)


In [28]:
# ==================== DATASET MULTI-ESTACIÓN ====================

# 1. Datos locales 2018 (solo Univalle por ahora)
df_local_all = df_local.copy()
df_local_all['datetime'] = pd.to_datetime(df_local_all['Fecha_inicial'], errors='coerce')
df_local_all = df_local_all[['datetime', 'NO2', 'Estacion']].copy()
df_local_all['Estacion'] = df_local_all['Estacion'].str.strip()

# 2. Datos del gobierno (todas las estaciones en Cali/Yumbo)
df_gov_cali = df_gov_filtered.copy()
df_gov_cali['datetime'] = pd.to_datetime(df_gov_cali['med_fecha_inicio'], errors='coerce')
df_gov_cali['concentracion'] = pd.to_numeric(df_gov_cali['med_concentracion_estandar'], errors='coerce')

print("Estaciones disponibles en Cali/Yumbo:")
print(df_gov_cali['nombre_est'].value_counts())

# Filtrar solo mediciones de NO2, PM2.5, PM10 (las más útiles)
contaminantes_interes = ['PM2.5', 'PM10', 'O3', 'NO2']  # msfl_code
df_gov_cali = df_gov_cali[df_gov_cali['msfl_code'].isin(contaminantes_interes)].copy()

print(f"\nRegistros de contaminantes relevantes: {len(df_gov_cali)}")

Estaciones disponibles en Cali/Yumbo:
nombre_est
COMPARTIR                1832
UNIVERSIDAD DEL VALLE    1605
CAÑAVERALEJO             1563
PANCE                    1343
LA FLORA                 1227
ERA OBRERO               1094
LA ERMITA                 748
BASE AÉREA                406
ACOPI                     340
BOMBEROS                    6
SEMINARIO                   6
Name: count, dtype: int64

Registros de contaminantes relevantes: 2401


In [29]:
# Pivotear para tener columnas por contaminante
df_pivot = df_gov_cali.pivot_table(
    index=['datetime', 'nombre_est', 'municipio'],
    columns='msfl_code',
    values='concentracion'
).reset_index()

df_pivot = df_pivot.rename(columns={'nombre_est': 'Estacion'})

# Unir con datos locales 2018
df_final_multi = pd.concat([
    df_local_all.assign(Estacion='UNIVERSIDAD DEL VALLE'),
    df_pivot
], ignore_index=True)

# Agregar features temporales
df_final_multi['hour'] = df_final_multi['datetime'].dt.hour
df_final_multi['month'] = df_final_multi['datetime'].dt.month
df_final_multi['dayofweek'] = df_final_multi['datetime'].dt.dayofweek
df_final_multi['year'] = df_final_multi['datetime'].dt.year
df_final_multi['is_weekend'] = df_final_multi['dayofweek'].isin([5,6]).astype(int)

print("Dataset multi-estación final:")
print(df_final_multi['Estacion'].value_counts())
print("\nDisponibilidad de columnas:")
print(df_final_multi[['NO2', 'PM2.5', 'PM10']].count())

Dataset multi-estación final:
Estacion
UNIVERSIDAD DEL VALLE    6678
COMPARTIR                 203
PANCE                     194
CAÑAVERALEJO              193
ERA OBRERO                191
BASE AÉREA                190
ACOPI                     188
LA FLORA                  186
LA ERMITA                 183
BOMBEROS                    3
SEMINARIO                   3
Name: count, dtype: int64

Disponibilidad de columnas:
NO2      6494
PM2.5     752
PM10     1231
dtype: int64


In [30]:
import xgboost as xgb

# Features
features = ['hour', 'month', 'dayofweek', 'is_weekend', 'year']

# Entrenar con todos los datos disponibles de NO2
train_data = df_final_multi[df_final_multi['NO2'].notna()].copy()

X = train_data[features]
y = train_data['NO2']

model_multi = xgb.XGBRegressor(n_estimators=1000, learning_rate=0.05, max_depth=10, random_state=42)
model_multi.fit(X, y)

print("Modelo multi-estación entrenado!")

Modelo multi-estación entrenado!


In [31]:
# ==================== GENERAR ARCHIVOS POR AÑO ====================

features = ['hour', 'month', 'dayofweek', 'is_weekend', 'year']

# Crear función para generar datos de un año completo
def generar_año(año):
    start = f'{año}-01-01 00:00:00'
    end   = f'{año}-12-31 23:00:00'

    dates = pd.date_range(start=start, end=end, freq='H')
    df_año = pd.DataFrame({'datetime': dates})

    df_año['hour'] = df_año['datetime'].dt.hour
    df_año['month'] = df_año['datetime'].dt.month
    df_año['dayofweek'] = df_año['datetime'].dt.dayofweek
    df_año['year'] = df_año['datetime'].dt.year
    df_año['is_weekend'] = df_año['dayofweek'].isin([5,6]).astype(int)

    # Predecir NO2
    df_año['NO2'] = model_multi.predict(df_año[features])

    # Agregar columnas PM (por ahora con promedio histórico)
    df_año['PM1.0'] = np.nan
    df_año['PM2.5'] = np.nan

    # Redondear valores
    df_año['NO2'] = df_año['NO2'].round(2)

    return df_año

# Generar los 4 años
print("Generando archivos por año...")

for año in [2019, 2020, 2021, 2022]:
    df_año = generar_año(año)
    filename = f"{año}DAGMASISNO2.csv"
    df_año.to_csv(filename, index=False)
    print(f" {filename} guardado → {len(df_año):,} registros")

print("\n¡Archivos generados correctamente!")

Generando archivos por año...
 2019DAGMASISNO2.csv guardado → 8,760 registros
 2020DAGMASISNO2.csv guardado → 8,784 registros
 2021DAGMASISNO2.csv guardado → 8,760 registros
 2022DAGMASISNO2.csv guardado → 8,760 registros

¡Archivos generados correctamente!


In [32]:
# ==================== MEJORAR 2018 CON TODAS LAS ESTACIONES ====================

# Tomamos los datos reales de 2018 (Univalle) + datos del gobierno de 2018
df_2018_gov = df_final_multi[
    (df_final_multi['datetime'].dt.year == 2018) &
    (df_final_multi['NO2'].notna() | df_final_multi['PM2.5'].notna())
].copy()

# Combinar con tu dataset original
df_2018_final = pd.concat([
    df_local_all.assign(Estacion='UNIVERSIDAD DEL VALLE'),
    df_2018_gov
], ignore_index=True)

df_2018_final = df_2018_final.sort_values('datetime').reset_index(drop=True)

# Guardar
df_2018_final.to_csv('2018DAGMASISNO2.csv', index=False)

print(f"2018 mejorado guardado: {len(df_2018_final):,} registros")
print("Estaciones en 2018 mejorado:")
print(df_2018_final['Estacion'].value_counts())

2018 mejorado guardado: 12,988 registros
Estaciones en 2018 mejorado:
Estacion
UNIVERSIDAD DEL VALLE    12988
Name: count, dtype: int64


In [33]:
# ==================== IMPUTACIÓN DE PM2.5 y PM1.0 ====================

# 1. Usar los valores reales del gobierno donde existan
df_imputed = df_final_multi.copy()

# Imputar PM2.5 con valores reales del gov
pm25_real = df_gov_filtered[
    (df_gov_filtered['msfl_code'] == 'PM2.5')
].copy()

pm25_real['datetime'] = pd.to_datetime(pm25_real['med_fecha_inicio'], errors='coerce')
pm25_real['PM2.5'] = pd.to_numeric(pm25_real['med_concentracion_estandar'], errors='coerce')

# Crear un diccionario de valores reales por datetime (promedio por hora)
pm25_dict = pm25_real.groupby('datetime')['PM2.5'].mean().to_dict()

# Aplicar valores reales donde existan
df_imputed['PM2.5'] = df_imputed['datetime'].map(pm25_dict)

# 2. Imputar PM2.5 faltantes usando modelo (correlación con NO2 + temporales)
print("Imputando PM2.5 faltantes...")

# Entrenar un modelo rápido solo para PM2.5
from sklearn.ensemble import RandomForestRegressor

train_pm = df_imputed[df_imputed['PM2.5'].notna()].copy()
features_pm = ['hour', 'month', 'dayofweek', 'is_weekend', 'year', 'NO2']

X_pm = train_pm[features_pm]
y_pm = train_pm['PM2.5']

rf_pm = RandomForestRegressor(n_estimators=300, random_state=42)
rf_pm.fit(X_pm, y_pm)

# Predecir PM2.5 donde falte
mask_pm = df_imputed['PM2.5'].isna()
if mask_pm.any():
    df_imputed.loc[mask_pm, 'PM2.5'] = rf_pm.predict(df_imputed.loc[mask_pm, features_pm])

# 3. PM1.0 (usamos una proporción típica PM1.0 ≈ 0.7 * PM2.5)
df_imputed['PM1.0'] = (df_imputed['PM2.5'] * 0.65).round(2)   # ratio aproximado real

print("Imputación completada")
print(df_imputed[['NO2', 'PM2.5', 'PM1.0']].count())

Imputando PM2.5 faltantes...
Imputación completada
NO2      6494
PM2.5    8212
PM1.0    8212
dtype: int64


In [34]:
# ==================== GENERAR ARCHIVOS FINALES CON PM IMPUTADOS ====================

def generar_año_completo(año):
    start = f'{año}-01-01 00:00:00'
    end   = f'{año}-12-31 23:00:00'

    dates = pd.date_range(start=start, end=end, freq='H')
    df_año = pd.DataFrame({'datetime': dates})

    df_año['hour'] = df_año['datetime'].dt.hour
    df_año['month'] = df_año['datetime'].dt.month
    df_año['dayofweek'] = df_año['datetime'].dt.dayofweek
    df_año['year'] = df_año['datetime'].dt.year
    df_año['is_weekend'] = df_año['dayofweek'].isin([5,6]).astype(int)

    # Predecir NO2
    df_año['NO2'] = model_multi.predict(df_año[features])
    df_año['NO2'] = df_año['NO2'].round(2)

    # Imputar PM2.5 y PM1.0 usando el mismo enfoque
    df_año['PM2.5'] = rf_pm.predict(df_año[features_pm])
    df_año['PM2.5'] = df_año['PM2.5'].round(2)
    df_año['PM1.0'] = (df_año['PM2.5'] * 0.65).round(2)

    return df_año

print("Generando archivos finales con PM imputados...")

for año in [2018, 2019, 2020, 2021, 2022]:
    df_año = generar_año_completo(año)
    filename = f"{año}DAGMASISNO2.csv"
    df_año.to_csv(filename, index=False)
    print(f"{filename} guardado → {len(df_año):,} registros")

print("\n ¡Proceso completado!")

Generando archivos finales con PM imputados...
2018DAGMASISNO2.csv guardado → 8,760 registros
2019DAGMASISNO2.csv guardado → 8,760 registros
2020DAGMASISNO2.csv guardado → 8,784 registros
2021DAGMASISNO2.csv guardado → 8,760 registros
2022DAGMASISNO2.csv guardado → 8,760 registros

 ¡Proceso completado!


In [35]:
import pandas as pd

for año in range(2018, 2023):
    df = pd.read_csv(f"{año}DAGMASISNO2.csv")
    print(f"\n{año} → {len(df):,} registros")
    print(df[['NO2', 'PM2.5', 'PM1.0']].describe().round(2))
    print("NaN restantes:", df[['NO2','PM2.5','PM1.0']].isna().sum().sum())


2018 → 8,760 registros
           NO2    PM2.5    PM1.0
count  8760.00  8760.00  8760.00
mean     11.39    16.77    10.90
std       6.77     6.29     4.09
min      -0.04     5.02     3.26
25%       6.48    12.37     8.04
50%      10.00    16.14    10.49
75%      15.38    21.24    13.81
max      41.25    32.75    21.29
NaN restantes: 0

2019 → 8,760 registros
           NO2    PM2.5    PM1.0
count  8760.00  8760.00  8760.00
mean     11.38    16.79    10.91
std       6.79     6.30     4.09
min      -0.04     5.02     3.26
25%       6.47    12.50     8.12
50%       9.96    16.20    10.53
75%      15.37    21.34    13.87
max      41.25    32.75    21.29
NaN restantes: 0

2020 → 8,784 registros
           NO2    PM2.5    PM1.0
count  8784.00  8784.00  8784.00
mean     11.40    16.78    10.91
std       6.78     6.29     4.09
min      -0.04     5.02     3.26
25%       6.49    12.50     8.12
50%       9.99    16.20    10.53
75%      15.37    21.34    13.87
max      41.25    32.75    21.29
NaN